# Atelier Préparation de Données Images

**Contexte.** Une entreprise souhaite entraîner un modèle de Machine Learning / Deep Learning capable de reconnaître automatiquement le type de déchet présent sur une photographie (`cardboard`, `glass`, `metal`, `paper`, `plastic`, `trash`) afin d'améliorer le tri des déchets. Les images collectées proviennent de plusieurs sources et ne sont donc pas homogènes (dimensions, formats, modes couleur, qualité). L'objectif de cet atelier est de construire, à partir du dossier `data/raw/`, un jeu de données images propre et homogène (`data/cleaned/`), prêt à être utilisé pour l'entraînement.

## Imports et configuration

In [1]:
import os
import shutil
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, UnidentifiedImageError

RAW_DIR = Path("../data/raw")
CLEANED_DIR = Path("../data/cleaned")
REPORTS_DIR = Path("../reports")
REPORTS_DIR.mkdir(exist_ok=True)

CLASSES = sorted([d.name for d in RAW_DIR.iterdir() if d.is_dir()])
CLASSES

['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']

## Partie 1 – Exploration du dataset

### Fonction d'extraction des métadonnées d'une image

In [2]:
def extraire_metadonnees(chemin: Path, classe: str) -> dict:
    """Extrait nom, classe, format, mode, largeur, hauteur, ecart-type des pixels,
    nombre de canaux et taille (octets) d'une image. Robuste aux fichiers corrompus :
    les champs dependant du contenu de l'image restent a None si la lecture echoue."""
    infos = {
        'nom': chemin.name,
        'classe': classe,
        'chemin': str(chemin),
        'taille_octets': chemin.stat().st_size,
        'format': None,
        'mode': None,
        'largeur': None,
        'hauteur': None,
        'ecart_type_pixels': None,
        'canaux': None,
    }
    try:
        with Image.open(chemin) as img:
            img.load()  # force le decodage complet : declenche une erreur si le fichier est corrompu
            infos['format'] = img.format
            infos['mode'] = img.mode
            infos['largeur'], infos['hauteur'] = img.size
            infos['canaux'] = len(img.getbands())
            infos['ecart_type_pixels'] = float(np.array(img).std())
    except Exception:
        pass
    return infos

### Construction du DataFrame d'audit sur l'ensemble du dataset (`data/raw/`)

In [3]:
lignes = []
for classe in CLASSES:
    for chemin in sorted((RAW_DIR / classe).iterdir()):
        if chemin.is_file():
            lignes.append(extraire_metadonnees(chemin, classe))

df_audit = pd.DataFrame(lignes)
df_audit['resolution'] = list(zip(df_audit['largeur'], df_audit['hauteur']))
print(f"Nombre total d'images dans data/raw/ : {len(df_audit)}")
df_audit.shape

Nombre total d'images dans data/raw/ : 1032


(1032, 11)